# Task 3: Multimodal ML – Housing Price Prediction Using Images + Tabular Data
This notebook builds a late-fusion multimodal neural network using PyTorch. The model processes house images through a Convolutional Neural Network (CNN) branch and structural house metrics through a Multi-Layer Perceptron (MLP) branch, combining their features to predict continuous housing prices.

# Objectives:
Generate / Preprocess a multimodal dataset containing both tabular properties and image data.

Build a CNN branch for visual feature extraction.

Build an MLP branch for structural feature extraction.

Concatenate both feature streams and train an end-to-end regression network.

Evaluate performance using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE).

In [6]:
# Install required libraries
!pip install -q torch torchvision pandas scikit-learn pillow matplotlib

# 1. Setup & Synthetic Multimodal Data Generation
Since working with real image paths can be messy across different environments, this block creates a local dataset structure on the fly with 500 samples. Each sample contains structural data (Bedrooms, Bathrooms, SqFt, Age) and a corresponding generated synthetic image representing a house thumbnail.

In [7]:
import os
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

# Create directories for the images
os.makedirs("./housing_images", exist_ok=True)

# Define dataset size
num_samples = 500
np.random.seed(42)

# 1. Generate Tabular Data
bedrooms = np.random.randint(1, 6, size=num_samples)
bathrooms = np.random.randint(1, 4, size=num_samples)
sqft = np.random.randint(800, 4000, size=num_samples)
age = np.random.randint(0, 50, size=num_samples)

# 2. Generate Synthetic Images and calculate a continuous Target Price
image_paths = []
prices = []

for i in range(num_samples):
    # Base price calculation determined by tabular features + random noise
    base_price = (bedrooms[i] * 50000) + (bathrooms[i] * 30000) + (sqft[i] * 150) - (age[i] * 1000)

    # Generate an image with variations based on house characteristics
    img = Image.new("RGB", (64, 64), color=(200, 220, 240))
    draw = ImageDraw.Draw(img)

    # Add a primitive "house shape" (roof and body) that scales loosely with SqFt
    size_factor = int((sqft[i] / 4000) * 20)
    draw.rectangle([20 - size_factor//2, 35, 44 + size_factor//2, 60], fill=(139, 69, 19)) # House body
    draw.polygon([(15, 35), (32, 15), (49, 35)], fill=(205, 92, 92)) # Roof

    # Target price shifts slightly based on the visual "complexity" or color variance
    img_path = f"./housing_images/house_{i}.jpg"
    img.save(img_path)
    image_paths.append(img_path)

    # Final price with added variation noise
    final_price = base_price + np.random.normal(0, 15000)
    prices.append(max(final_price, 50000)) # Floor price at $50k

# Combine into a clean Pandas DataFrame
df = pd.DataFrame({
    'Bedrooms': bedrooms,
    'Bathrooms': bathrooms,
    'SqFt': sqft,
    'Age': age,
    'Image_Path': image_paths,
    'Price': prices
})

print("Sample of the generated tabular data framework:")
print(df.head())

Sample of the generated tabular data framework:
   Bedrooms  Bathrooms  SqFt  Age                    Image_Path          Price
0         4          2  3360   12  ./housing_images/house_0.jpg  756857.001914
1         5          1  2119   12  ./housing_images/house_1.jpg  581477.650315
2         3          2  3765   17  ./housing_images/house_2.jpg  732992.631849
3         5          1  2655   31  ./housing_images/house_3.jpg  646493.657224
4         5          3  2172   31  ./housing_images/house_4.jpg  632449.044027


# 2. Define Custom PyTorch Dataset and Preprocessing
We build a robust PyTorch Dataset that handles normalisation for tabular features, applies spatial standardisation transforms to the image inputs, and extracts the numeric target variable.

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Train-Test Split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Fit scaler on tabular features
scaler = StandardScaler()
X_train_tab = scaler.fit_transform(train_df[['Bedrooms', 'Bathrooms', 'SqFt', 'Age']])
X_test_tab = scaler.transform(test_df[['Bedrooms', 'Bathrooms', 'SqFt', 'Age']])

# Define image transformation pipeline
image_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Custom Multimodal Dataset class
class MultimodalHousingDataset(Dataset):
    def __init__(self, dataframe, scaled_tabular, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.tabular_features = torch.tensor(scaled_tabular, dtype=torch.float32)
        self.labels = torch.tensor(self.df['Price'].values, dtype=torch.float32)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Load and transform image
        img_path = self.df.loc[idx, 'Image_Path']
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        # Get tabular vectors and corresponding label
        tabular = self.tabular_features[idx]
        label = self.labels[idx]

        return image, tabular, label

# Create DataLoaders
train_dataset = MultimodalHousingDataset(train_df, X_train_tab, transform=image_transforms)
test_dataset = MultimodalHousingDataset(test_df, X_test_tab, transform=image_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"DataLoaders successfully initiated. Training batches: {len(train_loader)}")

DataLoaders successfully initiated. Training batches: 13


# 3. Construct the Multimodal Architecture
We create a network using a late-fusion model design pattern:

CNN Branch: Extracts features from visual spatial frames down to a linear representation.

Tabular Branch: A simple deep dense layer configuration processing properties.

Fusion Layer: Concatenates outputs from both branches into final dense layers for regression output.

In [9]:
import torch.nn as nn
import torch.nn.functional as F

class MultimodalRegressionNet(nn.Module):
    def __init__(self, num_tabular_features):
        super(MultimodalRegressionNet, self).__init__()

        # 1. CNN Branch for Image Inputs
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Output: 16 x 32 x 32

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Output: 32 x 16 x 16

            nn.Flatten(),
            nn.Linear(32 * 16 * 16, 64),
            nn.ReLU()
        )

        # 2. MLP Branch for Tabular Inputs
        self.mlp = nn.Sequential(
            nn.Linear(num_tabular_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

        # 3. Fusion Layer Network (64 from CNN + 16 from MLP = 80 total inputs)
        self.fc_fusion = nn.Sequential(
            nn.Linear(64 + 16, 32),
            nn.ReLU(),
            nn.Linear(32, 1) # Continuous Price Output
        )

    def forward(self, img, tab):
        cnn_out = self.cnn(img)
        mlp_out = self.mlp(tab)

        # Fuse vectors across features dimension
        fused = torch.cat((cnn_out, mlp_out), dim=1)
        output = self.fc_fusion(fused)
        return output.squeeze(-1)

# Check device and instantiate
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultimodalRegressionNet(num_tabular_features=4).to(device)
print(model)

MultimodalRegressionNet(
  (cnn): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=8192, out_features=64, bias=True)
    (8): ReLU()
  )
  (mlp): Sequential(
    (0): Linear(in_features=4, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
  )
  (fc_fusion): Sequential(
    (0): Linear(in_features=80, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)


# 4. Train the Multimodal Model
We use Mean Squared Error (MSE) loss during training optimization alongside the Adam optimizer.

In [10]:
import torch.optim as optimizer_lib

criterion = nn.MSELoss()
optimizer = optimizer_lib.Adam(model.parameters(), lr=0.005)

epochs = 20
print(f"Beginning training loop execution on device: {device}...")

model.train()
for epoch in range(epochs):
    running_loss = 0.0
    for images, tabulars, labels in train_loader:
        images, tabulars, labels = images.to(device), tabulars.to(device), labels.to(device)

        # Forward pass
        optimizer.zero_grad()
        predictions = model(images, tabulars)
        loss = criterion(predictions, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_rmse = np.sqrt(running_loss / len(train_loader.dataset))
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{epochs} | Training Loss (RMSE value equivalent): ${epoch_rmse:.2f}")

Beginning training loop execution on device: cuda...
Epoch 1/20 | Training Loss (RMSE value equivalent): $566096.40
Epoch 5/20 | Training Loss (RMSE value equivalent): $171748.88
Epoch 10/20 | Training Loss (RMSE value equivalent): $144063.44
Epoch 15/20 | Training Loss (RMSE value equivalent): $83863.99
Epoch 20/20 | Training Loss (RMSE value equivalent): $74438.78


# 5. Model Evaluation using MAE and RMSE
We shift the network into evaluation status (model.eval()) and collect evaluation predictions to map metrics using scikit-learn.

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

model.eval()
all_predictions = []
all_ground_truths = []

with torch.no_grad():
    for images, tabulars, labels in test_loader:
        images, tabulars = images.to(device), tabulars.to(device)
        preds = model(images, tabulars)

        all_predictions.extend(preds.cpu().numpy())
        all_ground_truths.extend(labels.numpy())

# Convert arrays to numpy
all_predictions = np.array(all_predictions)
all_ground_truths = np.array(all_ground_truths)

# Evaluate metrics
mae = mean_absolute_error(all_ground_truths, all_predictions)
mse = mean_squared_error(all_ground_truths, all_predictions)
rmse = np.sqrt(mse)

print("\n=== Model Final Evaluation Performance ===")
print(f"Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")

# Compare structural baseline error to standard target range scale
target_range = all_ground_truths.max() - all_ground_truths.min()
print(f"Total Target Price Range across dataset: ${target_range:,.2f}")


=== Model Final Evaluation Performance ===
Mean Absolute Error (MAE): $65,453.54
Root Mean Squared Error (RMSE): $77,628.62
Total Target Price Range across dataset: $688,036.38
